# Tutorial 10: Quantum Advantage -- Resource Estimates and Honest Outlook

This notebook provides an honest assessment of quantum advantage for finance:
- Resource estimates for useful quantum speedups
- Break-even analysis (when does quantum beat classical?)
- Current hardware limitations
- Near-term and long-term outlook

**References**: Chakrabarti et al. (2021), Herman et al. (2023).

In [ ]:
import numpy as np
np.random.seed(42)

## 1. The Theoretical Promise

Quantum amplitude estimation provides a quadratic speedup over classical
Monte Carlo: $O(1/\epsilon)$ vs $O(1/\epsilon^2)$ queries to achieve error $\epsilon$.

For option pricing, Montanaro (2015) showed quantum Monte Carlo achieves
the same error as classical MC with quadratically fewer samples.

In [ ]:
# Classical MC: samples needed for error epsilon
def classical_samples(epsilon, sigma=1.0):
    """N = sigma^2 / epsilon^2"""
    return (sigma / epsilon) ** 2

# Quantum AE: Grover iterations for error epsilon
def quantum_queries(epsilon):
    """Q = O(1/epsilon)"""
    return 1.0 / epsilon

epsilons = [0.1, 0.01, 0.001, 0.0001]

print(f"{'Epsilon':>10} {'Classical N':>15} {'Quantum Q':>15} {'Speedup':>10}")
print("-" * 55)
for eps in epsilons:
    cn = classical_samples(eps)
    qq = quantum_queries(eps)
    print(f"{eps:>10.4f} {cn:>15.0f} {qq:>15.0f} {cn/qq:>10.0f}x")

## 2. The Overhead Reality

Each quantum query requires a full Grover iteration, which involves:
- State preparation (loading the distribution)
- Oracle (marking payoff above threshold)
- Diffusion operator

The constant factor matters enormously.

In [ ]:
# Realistic circuit parameters (from Chakrabarti et al. 2021)
# For a 20-qubit European option pricing circuit:
n_qubits = 20
gates_per_grover = 5000       # Approximate T-gate count per Grover iteration
t_gate_time_ns = 1            # Hypothetical: 1ns per logical T-gate (fault-tolerant)
classical_sample_time_ns = 10 # Classical MC: ~10ns per sample on modern CPU

print("Break-even analysis (Chakrabarti et al. 2021 estimates):")
print(f"  Qubits needed: ~{n_qubits}")
print(f"  T-gates per Grover iteration: ~{gates_per_grover:,}")
print()

for eps in [0.01, 0.001, 0.0001]:
    n_classical = classical_samples(eps)
    t_classical = n_classical * classical_sample_time_ns * 1e-9  # seconds
    
    n_grover = quantum_queries(eps)
    t_quantum = n_grover * gates_per_grover * t_gate_time_ns * 1e-9  # seconds
    
    ratio = t_classical / t_quantum if t_quantum > 0 else float('inf')
    winner = "Quantum" if ratio > 1 else "Classical"
    print(f"  eps={eps}: Classical={t_classical:.2f}s, Quantum={t_quantum:.4f}s, Ratio={ratio:.1f}x -> {winner}")

## 3. Current Hardware Limitations

In [ ]:
# Current (2025-2026) hardware parameters
hardware_params = {
    "IBM Eagle r3 (127q)": {
        "qubits": 127,
        "T1_us": 300,
        "T2_us": 200,
        "cx_error": 0.008,
        "readout_error": 0.012,
        "cx_time_ns": 660,
    },
    "IBM Heron r1 (156q)": {
        "qubits": 156,
        "T1_us": 350,
        "T2_us": 250,
        "cx_error": 0.005,
        "readout_error": 0.008,
        "cx_time_ns": 84,  # ECR gate
    },
}

for name, params in hardware_params.items():
    # Max circuit depth before decoherence dominates
    max_depth = int(params["T2_us"] * 1000 / params["cx_time_ns"])
    # Max meaningful Grover iterations
    gates_per_grover_est = 50  # For a 5-qubit toy problem
    max_grover = max_depth // gates_per_grover_est
    
    print(f"{name}:")
    print(f"  Max circuit depth (before T2): ~{max_depth}")
    print(f"  Max Grover iterations (5q toy): ~{max_grover}")
    print(f"  Achievable epsilon: ~{1.0/max_grover:.4f}" if max_grover > 0 else "  N/A")
    print()

## 4. qufin Head-to-Head: Quantum vs Classical

In [ ]:
import time
from qufin.options.classical.black_scholes import bs_price
from qufin.options.classical.monte_carlo import european_mc
from qufin.options.amplitude_estimation.european_qae import (
    EuropeanQAESpec, build_european_estimation_problem,
)
from qufin.options.amplitude_estimation.iqae import (
    IQAEConfig, IterativeAmplitudeEstimation,
)
from qufin.backends.qiskit_backend import QiskitAerBackend

S, K, sigma, r, T = 100, 105, 0.2, 0.05, 1.0
bs_ref = bs_price(s=S, k=K, sigma=sigma, r=r, T=T, option_type="call")

# Classical MC timing
t0 = time.time()
mc_price = european_mc(s=S, k=K, sigma=sigma, r=r, T=T, n_paths=100_000, option_type="call")
t_mc = time.time() - t0

# Quantum IQAE timing (simulator)
backend = QiskitAerBackend(shots=4096)
spec = EuropeanQAESpec(s=S, k=K, sigma=sigma, r=r, T=T, n_qubits=5, option_type="call")
problem = build_european_estimation_problem(spec)

t0 = time.time()
iqae = IterativeAmplitudeEstimation(
    problem=problem, backend=backend,
    config=IQAEConfig(epsilon_target=0.01),
)
qae_result = iqae.estimate()
t_qae = time.time() - t0

print(f"{'Method':<15} {'Price':>8} {'Error':>8} {'Time':>10}")
print("-" * 44)
print(f"{'BS (exact)':<15} {bs_ref:>8.4f} {0.0:>8.4f} {'<1ms':>10}")
print(f"{'MC (100K)':<15} {mc_price:>8.4f} {abs(mc_price-bs_ref):>8.4f} {t_mc:>9.3f}s")
print(f"{'IQAE (sim)':<15} {qae_result.value:>8.4f} {abs(qae_result.value-bs_ref):>8.4f} {t_qae:>9.3f}s")

## 5. When Will Quantum Win?

Based on Chakrabarti et al. (2021), quantum advantage for option pricing requires:
- ~7,500 logical qubits
- ~10^46 T-gates for a single Grover iteration at useful precision
- Error rates ~10^-10 (fault-tolerant)

This is far beyond current hardware, but the gap is closing.

In [ ]:
# Optimistic timeline estimates
milestones = [
    ("Toy demos (5-10 qubits, no advantage)", "2020-2025", "Achieved"),
    ("Noisy intermediate-scale (50-100 qubits)", "2025-2028", "In progress"),
    ("Early fault-tolerant (100-1000 logical qubits)", "2028-2032", "Projected"),
    ("Useful quantum advantage for pricing", "2032-2040", "Speculative"),
    ("Quantum advantage for portfolio opt.", "2030-2035", "Speculative"),
]

print(f"{'Milestone':<50} {'Timeline':<15} {'Status':<12}")
print("-" * 80)
for milestone, timeline, status in milestones:
    print(f"{milestone:<50} {timeline:<15} {status:<12}")

## 6. What qufin Enables Today

Even without quantum advantage, qufin provides value:

1. **Algorithm development**: Prototype and test quantum finance algorithms on simulators
2. **Benchmarking**: Quantify the gap between quantum and classical on standardized problems
3. **Hardware readiness**: When fault-tolerant hardware arrives, algorithms are ready
4. **Education**: Understand quantum algorithms through working code
5. **Research**: Reproduce and extend published results

In [ ]:
# qufin's honest benchmarking: let the data speak
from qufin.benchmarks.problems import make_benchmark

problem = make_benchmark(15)
print(f"Benchmark: {problem.n_assets} assets")
print(f"  Expected returns range: [{problem.mu.min():.4f}, {problem.mu.max():.4f}]")
print(f"  Covariance matrix rank: {np.linalg.matrix_rank(problem.sigma)}")
print()
print("The honest conclusion: quantum finance algorithms are not yet")
print("competitive with classical methods on any practical metric.")
print("But the theoretical foundations are sound, and the gap will close.")
print("qufin ensures you are ready when it does.")

## Summary

Key takeaways:

1. Quantum amplitude estimation offers a provable quadratic speedup -- but the constant
   factors are enormous.
2. Current hardware (2025-2026) supports only toy demonstrations (5-10 qubits).
3. Useful quantum advantage for finance likely requires fault-tolerant hardware
   with thousands of logical qubits.
4. Near-term value lies in algorithm development, benchmarking, and hardware readiness.
5. qufin provides honest, data-driven comparison -- if classical wins, the data shows it.

**This concludes the qufin tutorial series.**